In [1]:
import os

import pandas as pd

REPO_ROOT = os.path.dirname(os.getcwd())
DATA_PATH = os.path.join(REPO_ROOT, "dataset", "labels", "eligibility_annotations.csv")

df = pd.read_csv(DATA_PATH)
df.shape

(292, 6)

In [2]:
df["label"].value_counts()

label
OTHER_ELIGIBILITY    70
DISCIPLINE           60
NONE                 50
RESIDENCE            36
CAREER_STAGE         21
STUDENT_STATUS       17
NATIONALITY          15
EDUCATION            12
AGE                  11
Name: count, dtype: int64

In [3]:
df.groupby("opportunity_id").size().sort_values(ascending=False).head(10)

opportunity_id
mondriaan_fund_e52bef22781c                    22
national_culture_fund_bulgaria_8fca12fac738    15
kulturdirektoratet_norway_e1b48f36972d         14
dgartes_portugal_111bdc1e2ebf                  10
mondriaan_fund_a939eddff305                     9
creative_ireland_e896d7db1d6a                   9
dansateliers_rotterdam_e3404a6adf78             9
dansateliers_rotterdam_33931cad7e6f             8
kunsten_be_flanders_217b1f327dad                8
fpu_slovakia_c225bfd4c2a2                       7
dtype: int64

In [4]:
# 5 folds, grouped by opportunity so no call's chunks split across train/test
from sklearn.model_selection import StratifiedGroupKFold

X = df["chunk_text"]
y = df["label"]
groups = df["opportunity_id"]

# needs both: GroupKFold alone ignores the 9-class imbalance, StratifiedKFold alone
# would leak an opportunity's shared phrasing across train/test
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
all_labels = set(y.unique())

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
    train_groups = set(groups.iloc[train_idx])   # distinct opportunities in train
    test_groups = set(groups.iloc[test_idx])     # distinct opportunities in test
    overlap = train_groups & test_groups          # opportunities present on both sides
    assert not overlap, f"fold {fold}: group leakage on {overlap}"

    # smallest classes (AGE=11, EDUCATION=12) may still miss a fold's test split
    missing = all_labels - set(y.iloc[test_idx])
    print(f"fold {fold}: train={len(train_idx):3d}  test={len(test_idx):3d}  "
          f"missing_from_test={sorted(missing) or 'none'}")

fold 0: train=233  test= 59  missing_from_test=none
fold 1: train=235  test= 57  missing_from_test=none
fold 2: train=231  test= 61  missing_from_test=none
fold 3: train=236  test= 56  missing_from_test=none
fold 4: train=233  test= 59  missing_from_test=none


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

X_text = df[["chunk_text"]]  # DataFrame, not Series — ColumnTransformer needs a column name to select

baseline = Pipeline([
    # single-column selector: ColumnTransformer hands TfidfVectorizer a 1D array when given a bare column name
    ("features", ColumnTransformer([("tfidf", TfidfVectorizer(), "chunk_text")])),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

param_grid = {
    "features__tfidf__ngram_range": [(1, 1), (1, 2)],
    "features__tfidf__min_df": [1, 2, 3],
    "clf__C": [0.1, 1, 10],
}

# scoring=f1_macro, not accuracy — every class weighted equally despite the 6.5x imbalance
grid = GridSearchCV(baseline, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
grid.fit(X_text, y, groups=groups)

print("best params:", grid.best_params_)
print("best macro-F1 (mean over folds):", grid.best_score_)

best params: {'clf__C': 10, 'features__tfidf__min_df': 1, 'features__tfidf__ngram_range': (1, 1)}
best macro-F1 (mean over folds): 0.6250676215119533


In [7]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_predict

# out-of-fold predictions for the whole corpus, using the best hyperparameters found above
y_pred = cross_val_predict(grid.best_estimator_, X_text, y, cv=cv, groups=groups, n_jobs=-1)

print(classification_report(y, y_pred))

# ANNOTATION_GUIDELINES.md §2 menu order, for a readable confusion matrix
LABEL_ORDER = ["RESIDENCE", "NATIONALITY", "AGE", "DISCIPLINE", "CAREER_STAGE",
               "EDUCATION", "STUDENT_STATUS", "OTHER_ELIGIBILITY", "NONE"]

cm = confusion_matrix(y, y_pred, labels=LABEL_ORDER)
pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER)

                   precision    recall  f1-score   support

              AGE       1.00      0.82      0.90        11
     CAREER_STAGE       0.75      0.71      0.73        21
       DISCIPLINE       0.55      0.60      0.57        60
        EDUCATION       0.62      0.42      0.50        12
      NATIONALITY       0.67      0.53      0.59        15
             NONE       0.42      0.48      0.45        50
OTHER_ELIGIBILITY       0.58      0.64      0.61        70
        RESIDENCE       0.71      0.61      0.66        36
   STUDENT_STATUS       0.92      0.65      0.76        17

         accuracy                           0.60       292
        macro avg       0.69      0.61      0.64       292
     weighted avg       0.62      0.60      0.60       292



,RESIDENCE,NATIONALITY,AGE,DISCIPLINE,CAREER_STAGE,EDUCATION,STUDENT_STATUS,OTHER_ELIGIBILITY,NONE
RESIDENCE,22,0,0,4,1,0,0,5,4
NATIONALITY,1,8,0,3,0,0,0,1,2
AGE,0,0,9,0,0,0,0,1,1
DISCIPLINE,2,1,0,36,2,1,0,9,9
CAREER_STAGE,1,0,0,3,15,0,0,1,1
EDUCATION,0,0,0,3,1,5,1,1,1
STUDENT_STATUS,0,1,0,0,0,2,11,2,1
OTHER_ELIGIBILITY,3,1,0,7,0,0,0,45,14
NONE,2,1,0,10,1,0,0,12,24


In [8]:
# Known limitations of this TF-IDF baseline behind the NONE <-> DISCIPLINE errors
# (bare-heading chunks, a third cause, were removed from the data - ANNOTATION_GUIDELINES.md SS6):
#  1. arts vocabulary in non-constraint sentences - project/delivery descriptions
#     ("work with ... arts, cultural or creative organisations") are NONE but share
#     DISCIPLINE's words; bag-of-words can't tell "the project involves X" from
#     "the applicant must be X"
#  2. discipline terms too rare to learn - "early music", "costume designers",
#     "circus artists" appear once or twice in the corpus, and TF-IDF has no notion
#     that "dancers" ~ "performers", so an unseen term carries no signal -> NONE
# both motivate the sentence-embedding classifier; see flows.md, "Eligibility classifier"
# the confusion matrix's densest error cluster: DISCIPLINE / OTHER_ELIGIBILITY / NONE mixing with each other
confused = {"NONE", "DISCIPLINE", "OTHER_ELIGIBILITY"}
errors = df.assign(pred=y_pred)
errors = errors[(errors["label"] != errors["pred"])
                & errors["label"].isin(confused) & errors["pred"].isin(confused)]

for (true_label, pred_label), group in errors.groupby(["label", "pred"]):
    print(f"\n=== true={true_label}  predicted={pred_label}  (n={len(group)}) ===")
    for text in group["chunk_text"].head(3):
        print(f"  - {text}")


=== true=DISCIPLINE  predicted=NONE  (n=9) ===
  - We are inviting anyone to send us a proposal to create a performance, character or small happening around the (social) aspects of drinking tea (ceremonial, historical, colonial, its gossipy aspect etc) to be presented during our Dress-up-tea-Party.
  - For the ‘Dress-up’ part of our party, we welcome fashion- & costume designers to send a proposal of a selection of pieces/ costumes (from their oeuvre) to be used during the Dress-Up-Tea-Party.
  - Multidisciplinary projects are also welcome.

=== true=DISCIPLINE  predicted=OTHER_ELIGIBILITY  (n=9) ===
  - The Rencontres d’Arles curatorial research grant is aimed at exhibition curators wishing to carry out an original project relating to image or photography.
  - Marta & Kim are searching for a total of 2 additional performers, who meet the following conditions:
  - In addition, the embassy welcomes projects that explore innovative formats and presentation methods, cross-border cultural

In [9]:
pd.set_option("display.max_colwidth", 120)

# confident-error ranking: misclassifications where the model was most sure of a
# *different* label are the best candidates for a real annotation mistake, not
# just a hard/ambiguous chunk the model was unsure about
proba = cross_val_predict(grid.best_estimator_, X_text, y, cv=cv, groups=groups,
                           method="predict_proba", n_jobs=-1)
classes = grid.best_estimator_.classes_
confidence = proba.max(axis=1)
pred_label = classes[proba.argmax(axis=1)]

review = df.assign(pred=pred_label, confidence=confidence)
review = review[review["label"] != review["pred"]].sort_values("confidence", ascending=False)
review[["chunk_id", "chunk_text", "label", "pred", "confidence"]].head(20)

,chunk_id,chunk_text,label,pred,confidence
36,kunsten_be_flanders_1c78c30f66d3_0,"Eligible for support are Dutch-Belgian co-projects across all disciplines, but special attention is given to the per...",RESIDENCE,DISCIPLINE,0.808611
165,u_jazdowski_warsaw_ea8781efb99b_0,"Open call for Palestinian artists , cultural practitioners , theorists , curators , and people working at the inters...",NATIONALITY,DISCIPLINE,0.693019
221,kunst_dk_denmark_028cc37c2d3b_4,Events with audiences must be available to the public and they must be clearly announced.,NONE,OTHER_ELIGIBILITY,0.670228
66,dgartes_portugal_111bdc1e2ebf_6,"Entities must frame their activity in only one application and, in the event of support being granted, the respectiv...",NONE,OTHER_ELIGIBILITY,0.650486
57,kunsten_be_flanders_df27ab545c60_2,"We especially welcome applications from artists with migration backgrounds, artists of color, queer, neurodiverse, a...",NONE,NATIONALITY,0.615626
202,kulturdirektoratet_norway_90a7593f042e_3,"Individual artists, artist groups and individuals acting as independent actors in the field should instead apply to ...",OTHER_ELIGIBILITY,DISCIPLINE,0.589920
67,dgartes_portugal_111bdc1e2ebf_7,"103/2017, of 24 August, in its current wording.",NONE,RESIDENCE,0.583070
54,deutscher_uebersetzerfonds_c207b387a453_0,Work grant (target language German),DISCIPLINE,NONE,0.544200
167,u_jazdowski_warsaw_ea8781efb99b_2,"Target group Artists , cultural practitioners , theorists , curators , and people working at the intersection of edu...",NATIONALITY,DISCIPLINE,0.526933
58,dgartes_portugal_111bdc1e2ebf_0a,Private legal persons with headquarters in Portugal.,RESIDENCE,OTHER_ELIGIBILITY,0.518432


In [10]:
# ---- Model 2: Linear SVM ----
# Same pipeline and the same folds as the baseline - only the classifier changes,
# so any score difference comes from the model, not from the features or the split.
from sklearn.svm import LinearSVC

svm = Pipeline([
    ("features", ColumnTransformer([("tfidf", TfidfVectorizer(), "chunk_text")])),
    # LinearSVC instead of LogisticRegression: it looks for the widest margin between
    # classes rather than modelling probabilities - often a bit stronger on sparse,
    # high-dimensional TF-IDF text. Same class_weight="balanced" for the imbalance.
    # max_iter raised because large C values can otherwise stop before converging.
    ("clf", LinearSVC(class_weight="balanced", max_iter=10000)),
])

# same TF-IDF grid as the baseline; C gets one extra value on each side, because the
# baseline's best C=10 sat at the edge of its grid and the SVM's C scales differently.
# Edge check, run once outside this grid: the SVM's score is flat for C >= 100
# (0.643 at 100, 1000 and 10000 - with ~290 chunks and thousands of TF-IDF features the
# classes are already separable, so weaker regularisation stops changing anything),
# while LogReg peaks at C=10 and drops above it (0.623 at 100, 0.586 at 10000).
# So a best C on the grid edge here is a plateau, not a missed optimum.
svm_param_grid = {
    "features__tfidf__ngram_range": [(1, 1), (1, 2)],
    "features__tfidf__min_df": [1, 2, 3],
    "clf__C": [0.01, 0.1, 1, 10, 100],
}

svm_grid = GridSearchCV(svm, svm_param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
svm_grid.fit(X_text, y, groups=groups)

print("best params:", svm_grid.best_params_)
print("best macro-F1 (mean over folds):", svm_grid.best_score_)


best params: {'clf__C': 100, 'features__tfidf__min_df': 1, 'features__tfidf__ngram_range': (1, 1)}
best macro-F1 (mean over folds): 0.6430642222140884


In [11]:
# out-of-fold predictions with the best SVM, reported exactly like the baseline above
svm_pred = cross_val_predict(svm_grid.best_estimator_, X_text, y, cv=cv, groups=groups, n_jobs=-1)

print(classification_report(y, svm_pred))

svm_cm = confusion_matrix(y, svm_pred, labels=LABEL_ORDER)
pd.DataFrame(svm_cm, index=LABEL_ORDER, columns=LABEL_ORDER)


                   precision    recall  f1-score   support

              AGE       1.00      1.00      1.00        11
     CAREER_STAGE       0.75      0.71      0.73        21
       DISCIPLINE       0.54      0.57      0.55        60
        EDUCATION       0.71      0.42      0.53        12
      NATIONALITY       0.67      0.53      0.59        15
             NONE       0.45      0.48      0.47        50
OTHER_ELIGIBILITY       0.58      0.60      0.59        70
        RESIDENCE       0.65      0.67      0.66        36
   STUDENT_STATUS       0.88      0.82      0.85        17

         accuracy                           0.61       292
        macro avg       0.69      0.64      0.66       292
     weighted avg       0.61      0.61      0.61       292



,RESIDENCE,NATIONALITY,AGE,DISCIPLINE,CAREER_STAGE,EDUCATION,STUDENT_STATUS,OTHER_ELIGIBILITY,NONE
RESIDENCE,24,0,0,3,2,0,0,5,2
NATIONALITY,1,8,0,3,0,0,0,0,3
AGE,0,0,11,0,0,0,0,0,0
DISCIPLINE,3,1,0,34,3,0,0,9,10
CAREER_STAGE,0,0,0,3,15,0,0,2,1
EDUCATION,0,0,0,2,0,5,2,2,1
STUDENT_STATUS,0,1,0,0,0,1,14,0,1
OTHER_ELIGIBILITY,5,1,0,10,0,1,0,42,11
NONE,4,1,0,8,0,0,0,13,24


In [12]:
# ---- Baseline vs SVM, side by side ----
from sklearn.metrics import f1_score


def fold_scores(search):
    """Macro-F1 on each of the 5 test folds, for the best parameter setting of a GridSearchCV."""
    best = search.best_index_
    return [search.cv_results_[f"split{k}_test_score"][best] for k in range(cv.get_n_splits())]


# 1. per-fold macro-F1: both models saw identical folds, so fold k of one is directly
#    comparable to fold k of the other. With only ~60 test chunks per fold, a model
#    that wins on average but loses on most folds hasn't really won.
folds = pd.DataFrame({"baseline": fold_scores(grid), "svm": fold_scores(svm_grid)})
folds["svm - baseline"] = folds["svm"] - folds["baseline"]
print(folds.round(3), "\n")

# 2. per-class F1: shows *where* one model beats the other, not just whether
per_class = pd.DataFrame({
    "baseline": f1_score(y, y_pred, labels=LABEL_ORDER, average=None),
    "svm": f1_score(y, svm_pred, labels=LABEL_ORDER, average=None),
}, index=LABEL_ORDER)
per_class["svm - baseline"] = per_class["svm"] - per_class["baseline"]
print(per_class.round(2), "\n")

# 3. the NONE <-> DISCIPLINE confusion tracked since the baseline (see the limitations note above)
for name, pred in [("baseline", y_pred), ("svm", svm_pred)]:
    none_as_disc = ((y == "NONE") & (pred == "DISCIPLINE")).sum()
    disc_as_none = ((y == "DISCIPLINE") & (pred == "NONE")).sum()
    print(f"{name:8}  NONE->DISCIPLINE: {none_as_disc:2d}   DISCIPLINE->NONE: {disc_as_none:2d}")


   baseline    svm  svm - baseline
0     0.592  0.604           0.012
1     0.672  0.710           0.038
2     0.589  0.617           0.028
3     0.571  0.574           0.003
4     0.701  0.710           0.008 

                   baseline   svm  svm - baseline
RESIDENCE              0.66  0.66            0.00
NATIONALITY            0.59  0.59            0.00
AGE                    0.90  1.00            0.10
DISCIPLINE             0.57  0.55           -0.02
CAREER_STAGE           0.73  0.73            0.00
EDUCATION              0.50  0.53            0.03
STUDENT_STATUS         0.76  0.85            0.09
OTHER_ELIGIBILITY      0.61  0.59           -0.02
NONE                   0.45  0.47            0.02 

baseline  NONE->DISCIPLINE: 10   DISCIPLINE->NONE:  9
svm       NONE->DISCIPLINE:  8   DISCIPLINE->NONE: 10
